# Inspect HR snapshots

Lists `kbs-raw-snapshots`, prints the metadata on the Germany parental-leave policy, and compares the object count with `/ids` on the HR harness. Inside Compose the endpoint is `http://floci:4566`. On the host it is `http://127.0.0.1:4566`.

In [ ]:
import os

import boto3
import httpx
from botocore.config import Config

endpoint = os.environ.get("AWS_ENDPOINT_URL", "http://127.0.0.1:4566")
bucket = os.environ.get("RAW_BUCKET", "kbs-raw-snapshots")
harness = os.environ.get("HARNESS_HR_URL", "http://127.0.0.1:8101")
s3 = boto3.client(
    "s3",
    endpoint_url=endpoint,
    region_name=os.environ.get("AWS_DEFAULT_REGION", "us-east-1"),
    aws_access_key_id=os.environ.get("AWS_ACCESS_KEY_ID", "test"),
    aws_secret_access_key=os.environ.get("AWS_SECRET_ACCESS_KEY", "test"),
    config=Config(s3={"addressing_style": "path"}),
)
keys = []
token = None
while True:
    kwargs = {"Bucket": bucket, "Prefix": "raw/hr/"}
    if token:
        kwargs["ContinuationToken"] = token
    page = s3.list_objects_v2(**kwargs)
    keys.extend(item["Key"] for item in page.get("Contents", []))
    token = page.get("NextContinuationToken")
    if not token:
        break
print(len(keys), "snapshots")
key = "raw/hr/policy/pol-parental-leave-de.md"
head = s3.head_object(Bucket=bucket, Key=key)
print(key)
print(head["Metadata"])
body = s3.get_object(Bucket=bucket, Key=key)["Body"].read()
assert b"Elternzeit" in body
ids = httpx.get(f"{harness}/ids", timeout=120).json()
source = sum(len(rows) for rows in ids["ids"].values())
print("source", source, "snapshots", len(keys), "drift", len(keys) - source)
assert len(keys) == source
